# Case study: Travel booking A/B test

This notebook walks through one experiment end to end with `abkit`: trust checks first, then the primary result, guardrails, a Bayesian read-out, power, segments and an uplift model. The same pipeline runs for all four scenarios in `scripts/run_analysis.py`.

**Business problem.** Hotel booking completion is low: weak click-through from search results to booking pages, and high abandonment in checkout.

**Hypothesis.** *If* we show total price (incl. fees) on search results and collapse booking + payment into one page with express wallets, *then* booking conversion will increase, *because* price surprises and form length are the main causes of checkout abandonment.

> The dataset records which arm each session saw, not the exact design of B. The change described above is the hypothesised treatment used for this case study.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import pandas as pd
from abkit import data, analyze
from abkit.scenarios import get

pd.set_option("display.float_format", "{:.4f}".format)
sc = get("travel_booking")
df = data.load(sc)
df.head()

,SessionID,AB_Group,Device,UserType,PageViewed,Conversion,TimeOnPage_sec,Bounce,Revenue,group,treatment
0,1,A_Control,Desktop,New,Payment,0,81,0,0.0000,A,0
1,2,A_Control,Desktop,Returning,Booking,0,133,0,0.0000,A,0
2,3,B_Variation,Mobile,Returning,Payment,0,21,0,0.0000,B,1
3,4,B_Variation,Desktop,New,Payment,0,97,0,0.0000,B,1
4,5,B_Variation,Mobile,New,Search_Results,0,91,0,0.0000,B,1


## 1. Can we trust this experiment?

Before looking at any uplift, check the traffic split (sample ratio mismatch), duplicates, tracking consistency and whether the two arms look alike on attributes fixed *before* assignment.

In [2]:
res = analyze(sc, df=df)
pd.DataFrame(res["quality"]["checks"])[["name", "status", "detail"]]

,name,status,detail
0,Sample ratio mismatch,pass,"A=608 vs B=592 (B share 49.3%, expected 50%); ..."
1,Duplicate sessions,pass,0 duplicated SessionID values
2,Missing values,pass,0 null cells
3,Revenue / conversion consistency,pass,0 sessions with revenue but no conversion; 0 c...
4,Bounce / conversion consistency,pass,0 sessions marked as both bounced and converted
5,Bot / timer plausibility,pass,0 sessions with time on page <=2s or >1h (rang...
6,Pre-treatment covariate balance,warn,Largest imbalance: Device=Tablet (A 7.9% vs B ...
7,Page mix shift (post-treatment),info,Page mix differs between arms (p=0.009). Expec...


Device mix is slightly imbalanced (fewer tablet users in B). Device is a pre-treatment attribute, so that is worth a robustness check: re-estimate the effect while controlling for device and user type.

The page mix *also* differs (B has fewer sessions recorded on Payment). That is expected if B changes navigation, which is exactly why `PageViewed` is treated as post-treatment and is **not** used as a control variable.

In [3]:
res["adjusted"]

{'formula': 'Conversion ~ treatment + C(Device) + C(UserType)',
 'odds_ratio': 2.0135780803672843,
 'or_ci': (1.2864195006883101, 3.1517686754330154),
 'p_value': 0.0022006979503596067}

The adjusted odds ratio (~2.0, p≈0.002) matches the raw result, so the imbalance is not driving the finding.

## 2. Primary KPI: booking conversion

In [4]:
p = res["primary"]
print(f"A: {p['rate_a']:.2%}  ({p['x_a']}/{p['n_a']})")
print(f"B: {p['rate_b']:.2%}  ({p['x_b']}/{p['n_b']})")
print(f"Absolute lift: {p['abs_diff']*100:+.2f} pp  (95% CI {p['abs_ci'][0]*100:+.2f} to {p['abs_ci'][1]*100:+.2f})")
print(f"Uplift: {p['uplift']:+.1%}  (95% CI {p['uplift_ci'][0]:+.0%} to {p['uplift_ci'][1]:+.0%})")
print(f"z-test p = {p['p_value']:.4f}, Fisher exact p = {p['fisher_p']:.4f}")

A: 5.26%  (32/608)
B: 9.97%  (59/592)
Absolute lift: +4.70 pp  (95% CI +1.71 to +7.70)
Uplift: +89.4%  (95% CI +25% to +187%)
z-test p = 0.0021, Fisher exact p = 0.0022


![funnel](../reports/figures/travel_booking_funnel.png)

## 3. Secondary KPIs and guardrails

In [5]:
sec = res["secondary"]
pd.DataFrame({
    "metric": ["Revenue per visitor", "Avg booking value", "Time on page (s)"],
    "A": [sec["revenue_per_visitor"]["mean_a"], sec["avg_order_value"]["mean_a"], sec["time_on_page"]["mean_a"]],
    "B": [sec["revenue_per_visitor"]["mean_b"], sec["avg_order_value"]["mean_b"], sec["time_on_page"]["mean_b"]],
    "p": [sec["revenue_per_visitor"]["p_value"], sec["avg_order_value"]["p_value"], sec["time_on_page"]["p_value"]],
})

,metric,A,B,p
0,Revenue per visitor,48.4699,81.9352,0.0214
1,Avg booking value,920.9278,822.1292,0.2539
2,Time on page (s),86.4490,95.3412,0.0000


In [6]:
res["guardrails"]["bounce"]

{'rate_a': 0.34868421052631576,
 'rate_b': 0.25844594594594594,
 'abs_diff': -0.09023826458036982,
 'one_sided_bound': -0.04680473977159414,
 'margin': 0.02,
 'p_value': 1.4913106704197488e-05,
 'passed': True}

Revenue per visitor rises ~69% (bootstrap CI excludes zero), and bounce rate *falls*, so the guardrail passes comfortably.

One trade-off to watch: the average booking value is ~11% lower in B (not significant, p≈0.25). Transparent pricing may nudge people toward cheaper hotels. Revenue still rises because many more people book.

## 4. Bayesian read-out

Stakeholders rarely ask for a p-value. They ask *how likely is it that B is better, and what do we risk if we're wrong?*

In [7]:
b = res["bayes"]
print(f"P(B > A) = {b['prob_b_better']:.2%}")
print(f"Expected loss if we ship B = {b['expected_loss_b']*100:.5f} pp of conversion")
print(f"Posterior 95% interval for uplift: {b['rel_uplift_ci95'][0]:+.0%} to {b['rel_uplift_ci95'][1]:+.0%}")

P(B > A) = 99.91%
Expected loss if we ship B = 0.00039 pp of conversion
Posterior 95% interval for uplift: +25% to +186%


![posterior](../reports/figures/travel_booking_posterior.png)

## 5. Was the test big enough?

The effect was detected, but a significant result from a small test tends to *overstate* the true effect (the winner's curse).

In [8]:
pw = res["power"]
print(f"Baseline: {pw['baseline']:.2%}, n per arm: {pw['n_per_arm']}")
print(f"Minimum detectable effect at this n: {pw['mde_rel_at_n']:+.0%}")
print(f"n per arm to detect a 20% lift: {pw['n_for_target']:,}")
pw["winners_curse"]

Baseline: 5.26%, n per arm: 592
Minimum detectable effect at this n: +80%
n per arm to detect a 20% lift: 7,712


{'assumed_true_lift': 0.2,
 'power': 0.11624,
 'mean_significant_estimate': 0.8184259365528794,
 'exaggeration_ratio': 4.092129682764397}

At ~600 sessions per arm the test could only reliably detect a lift of about +80%; the observed +89% only just clears that bar. If the true lift were a more typical 20%, the test would have ~12% power, and the significant estimates it produced would be inflated about 4x on average. So **+89% is an upper-end estimate**; plan the business case around the lower CI bound and re-measure after rollout.

![power](../reports/figures/power_curves.png)

### Why you must not peek

In [9]:
pk = res["peeking"]
print(f"Fixed horizon FPR: {pk['fixed_fpr']:.1%}")
print(f"Peeking {pk['looks']}x and stopping at first p<0.05: {pk['naive_fpr']:.1%}")
print(f"Calibrated per-look threshold p<{pk['corrected_nominal_alpha']:.4f}: {pk['corrected_fpr']:.1%}")

Fixed horizon FPR: 5.0%
Peeking 10x and stopping at first p<0.05: 19.4%
Calibrated per-look threshold p<0.0104: 5.0%


![peeking](../reports/figures/peeking_simulation.png)

## 6. Does B work better for some users?

Segment results are adjusted for multiple testing, and heterogeneity is judged by a formal interaction test rather than by eyeballing per-segment p-values.

In [10]:
seg = pd.DataFrame(res["segments"])
seg[["variable", "level", "kind", "n_a", "n_b", "rate_a", "rate_b", "abs_diff", "p_raw", "p_holm", "p_bh"]]

,variable,level,kind,n_a,n_b,rate_a,rate_b,abs_diff,p_raw,p_holm,p_bh
0,Device,Desktop,pre-treatment,271,251,0.0443,0.0916,0.0474,0.0356,0.2134,0.0800
1,Device,Mobile,pre-treatment,289,310,0.0484,0.1097,0.0612,0.0064,0.0580,0.0580
2,Device,Tablet,pre-treatment,48,31,0.1250,0.0645,-0.0605,0.4703,0.9406,0.5291
3,UserType,New,pre-treatment,404,392,0.0470,0.0893,0.0423,0.0234,0.1636,0.0701
4,UserType,Returning,pre-treatment,204,200,0.0637,0.1200,0.0563,0.0581,0.2903,0.1045
5,PageViewed,Booking,exploratory,147,153,0.0544,0.1111,0.0567,0.0948,0.3792,0.1422
6,PageViewed,Hotel_Details,exploratory,162,171,0.0432,0.0526,0.0094,0.8000,0.9406,0.8000
7,PageViewed,Payment,exploratory,120,74,0.0583,0.1757,0.1173,0.0138,0.1101,0.0619
8,PageViewed,Search_Results,exploratory,179,194,0.0559,0.1031,0.0472,0.1266,0.3797,0.1627


In [11]:
pd.DataFrame(res["heterogeneity"])

,covariate,lr_stat,df,p_value
0,Device,3.6206,2,0.1636
1,UserType,0.0003,1,0.9854


Tablet is the only segment where B looks worse, but with 79 sessions the interval is huge, and the Device interaction test is not significant (p≈0.16). No segment survives Holm correction. **There is no evidence that B should be targeted differently by device or user type.**

![segments](../reports/figures/travel_booking_segments.png)

## 7. Uplift model

A T-learner estimates how much B changes each visitor's conversion probability from pre-treatment features. The Qini curve checks whether ranking visitors by predicted uplift beats random targeting, using a permutation test as the bar.

In [12]:
u = res["uplift_model"]
print(u["learner"])
print(f"Qini coefficient {u['qini_coefficient']:.2f} vs random mean {u['perm_mean']:.2f}; permutation p = {u['perm_p_value']:.3f}")
pd.DataFrame(u["segments"])

T-learner (L2 logistic regression), 5-fold cross-fitted
Qini coefficient -0.37 vs random mean -0.16; permutation p = 0.547


,Device,UserType,p_control,p_variation,pred_uplift,obs_rate_a,obs_rate_b,n_a,n_b,obs_uplift
0,Mobile,Returning,0.0599,0.1299,0.0699,0.0476,0.1635,105,104,0.1158
1,Mobile,New,0.0433,0.0984,0.0550,0.0489,0.0825,184,206,0.0336
2,Desktop,Returning,0.0562,0.1091,0.0530,0.0805,0.0795,87,88,-0.0009
3,Desktop,New,0.0406,0.0822,0.0416,0.0272,0.0982,184,163,0.0710
4,Tablet,New,0.1030,0.0658,-0.0373,0.1389,0.0870,36,23,-0.0519
5,Tablet,Returning,0.1391,0.0878,-0.0513,0.0833,0.0000,12,8,-0.0833


![qini](../reports/figures/travel_booking_qini.png)

The model does no better than random. That is a useful negative result: personalising B by device or user type would add complexity without proven benefit, so **roll B out to everyone**.

## 8. Decision

In [13]:
d = res["decision"]
print(d["label"]); print()
for r in d["reasons"]: print("•", r)
print()
for c in d["caveats"]: print("⚠", c)

Ship B - phased rollout

• Primary KPI up +89% (p=0.002); P(B > A) = 99.9%.
• Bounce-rate guardrail passed the non-inferiority test.

⚠ Some pre-treatment imbalance detected; the covariate-adjusted model gives OR 2.01 (p=0.002), so the conclusion holds.
⚠ Average value per conversion fell 11% (921 -> 822, p=0.25). B may be pulling in lower-value conversions; monitor value mix after rollout.
⚠ Lift above 50% is unusually large for a UI change (Twyman's law). Expect a smaller effect at full rollout; re-measure at 4-6 weeks. At this sample size, if the true lift were 20%, significant results would overstate it 4.1x on average.
⚠ The test is below the 7,712-per-arm sample planned for a 20% lift, so the uplift estimate is imprecise (95% CI +25% to +187%).


In [14]:
imp = res["impact"]
print(f"Assuming {imp['monthly_sessions_assumed']:,} sessions/month (illustrative):")
print(f"  extra bookings/month: {imp['extra_conversions_per_month']:,.0f} (95% CI {imp['extra_conversions_ci'][0]:,.0f} to {imp['extra_conversions_ci'][1]:,.0f})")
print(f"  extra revenue/month:  {imp['extra_revenue_per_month']:,.0f} (95% CI {imp['extra_revenue_ci'][0]:,.0f} to {imp['extra_revenue_ci'][1]:,.0f})")

Assuming 30,000 sessions/month (illustrative):
  extra bookings/month: 1,411 (95% CI 512 to 2,310)
  extra revenue/month:  1,003,959 (95% CI 156,789 to 1,883,851)


## Recommendation

1. **Phased rollout:** 25% → 50% → 100% over three weeks, with automatic rollback if bounce rate rises more than 2 pp or checkout errors increase.
2. **Size the business case on the lower bound** of the CI, not the +89% headline. The winner's curse analysis says the true effect is probably smaller.
3. **Monitor booking value mix**: average booking value dipped in B. Track revenue per visitor, not just conversions.
4. **Post-launch success check at 4-6 weeks:** conversion still above the pre-test baseline + 1 pp, with no degradation of bounce or page speed. Use a 5% holdout to measure it cleanly.
5. **Follow-up test:** separate the two changes (price transparency vs one-page checkout) to learn which one drives the lift.